<a href="https://colab.research.google.com/github/Halace-cmd/RDL/blob/main/Excercise1_Gurbisz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Einleitung
In diesem Notebook werden drei Transformer-basierte NLP-Aufgaben anhand des Papers Attention Is All You Need untersucht. Der Fokus liegt auf dem Abschnitt zu Encoder and Decoder Stacks, Attention und Multi-Head Attention. Zuerst wird der ausgewählte Textabschnitt mit T5-small zusammengefasst, sowohl direkt als auch mit einem Chunking-Ansatz, um die Wirkung unterschiedlicher Methoden auf die Qualität der Summary zu vergleichen. Danach wird mit BERT large fine-tuned on SQuAD ein Question-Answering-Modell getestet, um zentrale Informationen direkt aus dem Text zu extrahieren. Im letzten Teil wird GPT-2 medium für Text Generation verwendet, um ausgewählte Ideen aus dem Originaltext weiterzuführen und die Wirkung verschiedener Generierungsparameter wie temperature, top_k und top_p zu analysieren. Abschließend werden die Ergebnisse hinsichtlich Genauigkeit, Vollständigkeit, Kohärenz und Modellgrenzen verglichen.

In [ ]:
!pip install "transformers<5" sentencepiece -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.2 MB/s eta 0:00:00


# 1. Text-Zusammenfassung

In [ ]:
# Falls nötig:
# !pip install "transformers<5" sentencepiece -q

from transformers import pipeline

# Modell laden
summarizer = pipeline("summarization", model="t5-small")

# Originaltext
text = encode_decode_attention_text_stripped = """
Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully connected feed-forward network. We employ a residual connection around each of the two sub-layers, followed by layer normalization. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512. Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head attention over the output of the encoder stack. Similar to the encoder, we employ residual connections around each of the sub-layers, followed by layer normalization. We also modify the self-attention sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This masking, combined with fact that the output embeddings are offset by one position, ensures that the predictions for position i can depend only on the known outputs at positions less than i.

An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key.

We call our particular attention "Scaled Dot-Product Attention". The input consists of queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the query with all keys, divide each by √ dk, and apply a softmax function to obtain the weights on the values. In practice, we compute the attention function on a set of queries simultaneously, packed together into a matrix Q. The keys and values are also packed together into matrices K and V. The two most commonly used attention functions are additive attention, and dot-product (multiplicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor of √ 1 dk . Additive attention computes the compatibility function using a feed-forward network with a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is much faster and more space-efficient in practice, since it can be implemented using highly optimized matrix multiplication code. While for small values of dk the two mechanisms perform similarly, additive attention outperforms dot product attention without scaling for larger values of dk. We suspect that for large values of dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradients. To counteract this effect, we scale the dot products by √ 1 dk .

Instead of performing a single attention function with dmodel-dimensional keys, values and queries, we found it beneficial to linearly project the queries, keys and values h times with different, learned linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional Multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. With a single attention head, averaging inhibits this. In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality.

The Transformer uses multi-head attention in three different ways:

In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder. This allows every position in the decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder attention mechanisms in sequence-to-sequence models such as.

The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries come from the same place, in this case, the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.

Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position. We need to prevent leftward information flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled dot-product attention by masking out (setting to −∞) all values in the input of the softmax which correspond to illegal connections.
"""

# Zusammenfassung erzeugen
summary = summarizer(
    "summarize: " + text,
    max_length=150,
    min_length=60,
    num_beams=6,
    length_penalty=1.5,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
    do_sample=False,
    truncation=True
)

print("Model Summary:\n")
print(summary[0]["summary_text"])

Device set to use cpu
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model Summary:

the encoder is composed of a stack of N = 6 identical layers . the output of each layer is LayerNorm(x + Sublayer(x)) the decoder inserts a third sub-layer, which performs multi-head attention . we call our particular attention "Scaled Dot-Product Attention"


Die Ergebnisse der Zusammenfassung zeigen, dass T5-small einige zentrale Konzepte aus dem Transformer-Abschnitt erkennen kann, aber Schwierigkeiten hat, dichten technischen Text vollständig und zusammenhängend zusammenzufassen. In der Version ohne Chunking erzeugte das Modell nur eine sehr kurze Summary, die zwar einige wichtige Ideen wie die Encoder-Struktur mit sechs Schichten, Layer Normalization und Scaled Dot-Product Attention erwähnt, aber viele wesentliche Aspekte auslässt. Dazu gehören unter anderem das Masking im Decoder, Residual Connections, die Rolle von Queries, Keys und Values sowie die verschiedenen Einsatzarten von Multi-Head Attention im Transformer.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# Dein Originaltext
full_text = text = """
Encoder and Decoder Stacks

Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers.
The first is a multi-head self-attention mechanism, and the second is a simple, positionwise fully
connected feed-forward network. We employ a residual connection around each of the two sub-layers,
followed by layer normalization. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)),
where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual
connections, all sub-layers in the model, as well as the embedding layers, produce outputs of
dimension dmodel = 512.

Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
attention over the output of the encoder stack. Similar to the encoder, we employ residual connections
around each of the sub-layers, followed by layer normalization. We also modify the self-attention
sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for position i can depend only on the known outputs at positions less than i.

Attention

An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
of the values, where the weight assigned to each value is computed by a compatibility function of the
query with the corresponding key.

Scaled Dot-Product Attention

We call our particular attention "Scaled Dot-Product Attention". The input consists of queries and
keys of dimension dk, and values of dimension dv. We compute the dot products of the query with
all keys, divide each by √dk, and apply a softmax function to obtain the weights on the values. In
practice, we compute the attention function on a set of queries simultaneously, packed together into
a matrix Q. The keys and values are also packed together into matrices K and V. The two most
commonly used attention functions are additive attention, and dot-product (multiplicative) attention.
Dot-product attention is identical to our algorithm, except for the scaling factor of √(1/dk).
Additive attention computes the compatibility function using a feed-forward network with a single
hidden layer. While the two are similar in theoretical complexity, dot-product attention is much
faster and more space-efficient in practice, since it can be implemented using highly optimized matrix
multiplication code. While for small values of dk the two mechanisms perform similarly, additive
attention outperforms dot product attention without scaling for larger values of dk. We suspect that
for large values of dk, the dot products grow large in magnitude, pushing the softmax function into
regions where it has extremely small gradients. To counteract this effect, we scale the dot products
by √(1/dk).

Multi-Head-Attention

Instead of performing a single attention function with dmodel-dimensional keys, values and queries,
we found it beneficial to linearly project the queries, keys and values h times with different, learned
linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of
queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional
outputs. Multi-head attention allows the model to jointly attend to information from different
representation subspaces at different positions. With a single attention head, averaging inhibits this.
In this work we employ h = 8 parallel attention layers, or heads. For each of these we use
dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost
is similar to that of single-head attention with full dimensionality.

Applications of Attention in our Model

The Transformer uses multi-head attention in three different ways:

In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the
memory keys and values come from the output of the encoder. This allows every position in the
decoder to attend over all positions in the input sequence. This mimics the typical encoder-decoder
attention mechanisms in sequence-to-sequence models such as.

The encoder contains self-attention layers. In a self-attention layer all of the keys, values and queries
come from the same place, in this case, the output of the previous layer in the encoder. Each position
in the encoder can attend to all positions in the previous layer of the encoder.

Similarly, self-attention layers in the decoder allow each position in the decoder to attend to all
positions in the decoder up to and including that position. We need to prevent leftward information
flow in the decoder to preserve the auto-regressive property. We implement this inside of scaled
dot-product attention by masking out (setting to −∞) all values in the input of the softmax which
correspond to illegal connections.
"""

# 1. Thematische Chunks
part1, rest = full_text.split("An attention function can be described as", 1)
part2_body, rest = rest.split('We call our particular attention "Scaled Dot-Product Attention".', 1)
part3_body, rest = rest.split("Instead of performing a single attention function", 1)
part4_body, part5_body = rest.split("The Transformer uses multi-head attention in three different ways:", 1)

chunk1 = part1.strip()
chunk2 = ("An attention function can be described as" + part2_body).strip()
chunk3 = ('We call our particular attention "Scaled Dot-Product Attention".' + part3_body).strip()
chunk4 = ("Instead of performing a single attention function" + part4_body).strip()
chunk5 = ("The Transformer uses multi-head attention in three different ways:" + part5_body).strip()

chunks = [chunk1, chunk2, chunk3, chunk4, chunk5]

# 2. Bessere Summarization-Funktion
def summarize_text(input_text, max_new_tokens=110, min_length=50):
    prompt = "summarize the following transformer architecture text clearly and completely: " + input_text

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    summary_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        min_length=min_length,
        num_beams=10,
        length_penalty=1.0,
        repetition_penalty=1.5,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# 3. Chunk-Summaries erzeugen
chunk_summaries = []

for i, chunk in enumerate(chunks, 1):
    summary = summarize_text(chunk, max_new_tokens=110, min_length=50)
    chunk_summaries.append(summary)

    print(f"\nSummary of chunk {i}:")
    print(summary)

# 4. Kombinieren
combined_summary = " ".join(chunk_summaries)

print("\nCombined Summary:\n")
print(combined_summary)

# 5. Finale Summary
final_summary = summarize_text(combined_summary, max_new_tokens=170, min_length=100)

print("\nFinal Summary:\n")
print(final_summary)


Summary of chunk 1:
the encoder is composed of a stack of N = 6 identical layers. each layer has two sub-layers. the first is a multi-head self-attention mechanism, and the second is an integrated feed-forward network.

Summary of chunk 2:
an attention function can be described as mapping a query and key-value pairs to an output, where the query, keys, values, and output are all vectors. the output is computed as a weighted sum of the values.

Summary of chunk 3:
additive attention is identical to our algorithm, except for the scaling factor of (1/dk) additive attention computes the compatibility function using a feed-forward network with a single hidden layer. additive attention outperforms dot product attention without scaling for larger values of dk.

Summary of chunk 4:
multi-head attention allows the model to jointly attend to information from different representation subspaces at different positions. with a single attention head, averaging inhibits this. for each of these we emp

Der Chunking-Ansatz hat das Ergebnis deutlich verbessert. Durch die Aufteilung des Textes in thematische Abschnitte konnte das Modell mehr Informationen zu Attention, additive versus dot-product attention und Multi-Head Attention erhalten. Das zeigt, dass Chunking eine sinnvolle Methode ist, wenn der Eingabetext für ein kleines Modell zu lang oder zu komplex ist. Trotzdem verlor auch die abschließende, automatisch erzeugte Gesamtsummary wieder wichtige Informationen und enthielt teils ungenaue Formulierungen wie „integrated feed-forward network“, was nicht der Terminologie des Originaltexts entspricht.

# 1.1 Analyse der Textzusammenfassung
Im Vergleich zur manuellen Zusammenfassung waren die Modellzusammenfassungen insgesamt ungenauer, unvollständiger und weniger gut strukturiert. Die manuelle Summary konnte die Beziehung zwischen Encoder-Decoder-Architektur, Scaled Dot-Product Attention und dem Masking im Decoder deutlich besser darstellen. Insgesamt zeigt dieses Experiment, dass T5-small zwar Kerngedanken erkennen kann, aber akademischen Text oft zu stark komprimiert und wichtige technische Details verliert. Die Chunking-Version war klar besser als die Zusammenfassung ohne Chunking und daher die stärkere Modelllösung.

# 2.QA Modell

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="bert-large-uncased-whole-word-masking-finetuned-squad"
)

# "How many identical layers does the encoder have?"
# "What is the first sub-layer in each encoder layer?"
# "What is the second sub-layer in each encoder layer?"
# "What does the third sub-layer in the decoder do?"
# "Why is masking used in the decoder?"
# "What can an attention function be described as mapping?"
# "How is the output of attention computed?"
# "What does scaled dot-product attention divide by?"
# "Why is scaling applied in dot-product attention?"
# "How many heads are used in multi-head attention?"
# "What dimensions are used for dk and dv in each head?"
# "Why is multi-head attention beneficial?"
# "In how many different ways does the Transformer use multi-head attention?"
# "Where do the queries come from in encoder-decoder attention?"
# "Where do the keys, values, and queries come from in encoder self-attention?"
# "What must be prevented in decoder self-attention to preserve the auto-regressive property?"

question = "How many identical layers does the encoder have?"
result = qa_pipeline(
    question=question,
    context=text
)

print("Answer:", result["answer"])
print("Score:", round(result["score"], 4))
print("Question:", question)

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Answer: N = 6
Score: 0.8941
Question: How many identical layers does the encoder have?


# 2.1 Analyse zum Question Answering
Das verwendete QA-Modell bert-large-uncased-whole-word-masking-finetuned-squad funktionierte gut bei faktischen Fragen, deren Antworten klar und direkt im Text enthalten waren. Zum Beispiel wurde die Frage „How many identical layers does the encoder have?“ korrekt mit „N = 6“ beantwortet, bei einem hohen Confidence Score von 0.8941. Das zeigt, dass das Modell gut darin ist, kurze und exakte Antwortspannen aus dem gegebenen Kontext zu extrahieren.

Besonders gut funktionierte das Modell bei Fragen, die sich auf klar identifizierbare Textstellen beziehen, etwa zur Anzahl der Schichten, zur ersten Sub-Layer des Encoders oder zur Grunddefinition von Attention. In solchen Fällen konnte das Modell die richtige Antwort mit guter Genauigkeit finden. Schwächer war die Leistung jedoch bei längeren oder technisch komplexeren Fragen, insbesondere wenn der Kontext sehr lang war oder die richtige Antwort eine längere Erklärung erforderte. In einem Fall wurde zum Beispiel nur „vectors“ als Antwort gegeben, obwohl die eigentliche Antwort deutlich länger und spezifischer war. Das zeigt, dass der Score allein nicht ausreicht und jede Antwort zusätzlich inhaltlich geprüft werden muss.

Insgesamt war das QA-Modell gut geeignet, um klare Fakten aus dem Text herauszuziehen. Seine Hauptgrenze lag darin, dass es bei komplexeren technischen Fragen manchmal nur unvollständige Antwortspannen auswählte. Daraus lässt sich schließen, dass das Modell für präzise Faktenextraktion gut funktioniert, aber bei tieferem technischen Verständnis oder längeren Erklärungen nur eingeschränkt zuverlässig ist, wenn der Kontext nicht gezielt eingegrenzt wird.

# 3. Text-Generation

In [ ]:
from transformers import pipeline

from transformers import pipeline

# Modell laden
generator = pipeline("text-generation", model="gpt2-medium")

prompt = """
Original text:
Scaled dot-product attention computes the dot products of the query with all keys, divides each
"""
# Drei verschiedene Parameter-Settings
settings = [
    {
        "name": "Low temperature",
        "max_new_tokens": 60,
        "temperature": 0.5,
        "top_k": 40,
        "top_p": 0.85,
        "repetition_penalty": 1.2
    },
    {
        "name": "Balanced",
        "max_new_tokens": 60,
        "temperature": 0.7,
        "top_k": 50,
        "top_p": 0.9,
        "repetition_penalty": 1.2
    },
    {
        "name": "More creative",
        "max_new_tokens": 60,
        "temperature": 0.9,
        "top_k": 80,
        "top_p": 0.95,
        "repetition_penalty": 1.1
    }
]

# Text generieren
for s in settings:
    print(f"\n--- {s['name']} ---")

    output = generator(
        prompt,
        max_new_tokens=s["max_new_tokens"],
        temperature=s["temperature"],
        top_k=s["top_k"],
        top_p=s["top_p"],
        repetition_penalty=s["repetition_penalty"],
        do_sample=True,
        truncation=True,
        num_return_sequences=1,
        pad_token_id=50256
    )

    print(output[0]["generated_text"])

Device set to use cpu



--- Low temperature ---

Original text:
Scaled dot-product attention computes the dot products of the query with all keys, divides each by sqrt(dk), and applies a softmax function to obtain the weights on the values.

Write a short explanation based on this original text:


--- Balanced ---

Original text:
Scaled dot-product attention computes the dot products of the query with all keys, divides each by sqrt(dk), and applies a softmax function to obtain the weights on the values.

Write a short explanation based on this original text:
 "The problem is that we want our queries in an environment where every key has 1 weight (for example) but not 2 or 3 - since they are often used together for simple computations." This may seem counterintuitive when you look at it from another perspective though because if two different people use identical

--- More creative ---

Original text:
Scaled dot-product attention computes the dot products of the query with all keys, divides each by sqrt(dk), 

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2-medium")

prompt = """
Original text:
Scaled dot-product attention computes the dot products of the query with all keys, divides each
"""

output = generator(
    prompt,
    max_new_tokens=50,
    temperature=0.5,
    top_k=40,
    top_p=0.85,
    repetition_penalty=1.2,
    do_sample=True,
    truncation=True,
    num_return_sequences=1,
    pad_token_id=50256
)

print(output[0]["generated_text"])

Device set to use cpu



Original text:
Scaled dot-product attention computes the dot products of the query with all keys, divides each
. The result is a scalar value that represents how much time it takes for an object to be processed in memory (in seconds). It can also represent some other metric such as CPU usage or disk I/O throughput when using multiples queries on


# 3.1 Analyse der Text-Generation
Für die Textgenerierung wurde gpt2-medium verwendet, um Ideen aus dem Originalabschnitt des Papers weiterzuführen oder zu erweitern. Die generierten Ausgaben zeigen, dass das Modell zwar flüssigen Text erzeugen kann, aber für eine fachlich korrekte Erweiterung wissenschaftlicher Inhalte nur eingeschränkt geeignet ist. Bei Prompts auf Basis des Originaltexts zu Scaled Dot-Product Attention begann der generierte Text teilweise noch thematisch passend, driftete jedoch schnell vom eigentlichen Inhalt ab. In mehreren Beispielen erfand das Modell zusätzliche technische Details, nutzte unpassende Beispiele oder erzeugte klar halluzinierte Inhalte, die im Originalpaper nicht vorkommen.

Auch der Vergleich der Parameter zeigte deutliche Unterschiede. Die Einstellung mit niedriger Temperatur erzeugte den stabilsten Output, blieb aber trotzdem oft ungenau oder unvollständig. Die balancierte Einstellung brachte etwas mehr Variation, führte aber auch zu klar falschen und irrelevanten Inhalten. Die kreativere Einstellung war am wenigsten zuverlässig, da sie sich am stärksten vom Ausgangstext entfernte und die meisten unplausiblen Aussagen erzeugte. Damit zeigt sich, dass mehr Kreativität durch höhere Temperatur und lockerere Sampling-Parameter zwar zu abwechslungsreicherem Text führt, aber die fachliche Genauigkeit und Relevanz deutlich verschlechtert.

Im Vergleich zum Originaltext des Papers waren die generierten Texte oft weniger präzise, weniger kohärent und technisch unzuverlässig. GPT-2 medium konnte die Form eines erklärenden Textes teilweise imitieren, hat die eigentliche Bedeutung des Referenztexts aber nicht verlässlich erhalten. Dadurch eignet sich das Modell gut, um den Einfluss von Parametern auf Textgenerierung zu untersuchen, aber nicht, um ohne menschliche Nachbearbeitung vertrauenswürdige Erweiterungen wissenschaftlicher Inhalte zu erzeugen. Das Experiment zeigt damit sehr klar sowohl die Stärken als auch die Schwächen generativer Modelle: Sie können flüssigen Text produzieren, halluzinieren aber leicht und verlieren schnell die Bindung an den Ausgangstext.

Kürzere Version

GPT-2 medium konnte flüssigen Text erzeugen, war aber nicht zuverlässig genug für eine fachlich korrekte Erweiterung des Paper-Inhalts. Die niedrigere Temperatur lieferte die stabilsten Ergebnisse, während höhere Temperaturen zu kreativeren, aber auch deutlich ungenaueren und irrelevanteren Outputs führten. In mehreren Fällen halluzinierte das Modell Inhalte und entfernte sich stark vom Originaltext. Das zeigt, dass GPT-2 medium gut geeignet ist, um Textgenerierungsverhalten zu untersuchen, seine Ergebnisse aber kritisch bewertet und korrigiert werden müssen.